# **Baseline Logistic Regression on REAL Data**
**Project:** BizFlow360 — ML Early Warning System for Kenyan MSMEs
**Author:** Edusei Mikel Lisamba (Team Lead & ML Integration)

**What this notebook does:** Trains the Logistic Regression BASELINE model on the unified REAL KNBS dataset (`unified_msme_modeling_data.csv`), evaluates it, and saves the model + preprocessors so the advanced models (notebooks 11-13) can be compared against it.

In [3]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

# Base directory = ml_models/ (one level up from notebooks_2/)
base_dir = os.path.abspath('..')
print(f"Base directory: {base_dir}")

Base directory: /home/mikel/BizFlow360/ml_models


# 1. Load the Unified Real Dataset
**What this cell does:** Loads the clean, merged KNBS dataset that we created in notebook 09 and checks the balance of our target variable (`distress_label`).

In [4]:
data_path = os.path.join(base_dir, 'data', 'unified_msme_modeling_data.csv')
df = pd.read_csv(data_path)

print(f"✅ Loaded unified REAL dataset: {df.shape}")
print(f"\nTarget distribution (0 = Stable, 1 = Distressed):")
print(df['distress_label'].value_counts())
df.head()

✅ Loaded unified REAL dataset: (15814, 23)

Target distribution (0 = Stable, 1 = Distressed):
distress_label
0    10039
1     5775
Name: count, dtype: int64


,county,sector,male_working_owners,female_working_owners,total_monthly_expenses,monthly_rent_expense,monthly_electricity_expense,monthly_credit_expense,monthly_social_responsibility_expense,revenue_last_month,...,stock_value_end,total_turnover_2015,net_income_margin,revenue_change_ratio,business_closed,number_closed_establishments,revenue_decline,zero_or_missing_net_income,low_revenue,distress_label
0,MIGORI,95 - Repair of computers and personal and hous...,1,0,570.00,300.0,240.00,0.0,0.0,0.0,...,0.0,288000.0,1.333333,0.000000,0,1.0,1,0,1,0
1,KIAMBU,"47 - Retail trade, except of motor vehicles an...",1,0,2242.50,1950.0,195.00,0.0,0.0,39000.0,...,10000.0,288000.0,5.128205,4.000000,0,1.0,0,0,0,1
2,KIAMBU,"47 - Retail trade, except of motor vehicles an...",0,1,2254.20,1950.0,97.50,0.0,0.0,6825.0,...,10000.0,288000.0,2.051282,1.000000,0,1.0,0,1,1,0
3,SIAYA,"47 - Retail trade, except of motor vehicles an...",1,0,5.85,0.0,0.00,0.0,0.0,2925.0,...,50000.0,288000.0,1.709402,1.000000,0,1.0,0,0,1,0
4,KAJIADO,"46 - Wholesale trade, except of motor vehicles...",0,1,10130.25,5850.0,243.75,0.0,0.0,133380.0,...,500000.0,288000.0,0.946746,1.052308,0,1.0,0,0,0,0


# 2. Safety Cleaning (KNBS Placeholders)
**What this cell does:** A final safety net. If any KNBS "missing" codes (like -19.11 or -19.305) survived the merge, this cell converts them to NaN and fills them with the median so the model never sees fake negative numbers.

In [5]:
knbs_placeholders = [-19.11, -19.305, -2.4696, -2.4948, -20.58, -73.5, -74.25, -3.528, -8.82]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    df[col] = df[col].replace(knbs_placeholders, np.nan)
    df[col] = df[col].fillna(df[col].median())

print(f"✅ Safety cleaning done. Remaining NaNs: {df.isnull().sum().sum()}")

✅ Safety cleaning done. Remaining NaNs: 0


# 3. Encode Categorical Variables
**What this cell does:** Converts the text columns (`county` and `sector`) into numbers using Label Encoding, so the Logistic Regression model can understand them.

In [6]:
le_county = LabelEncoder()
le_sector = LabelEncoder()

df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

print(f"✅ Encoded {len(le_county.classes_)} counties and {len(le_sector.classes_)} sectors.")

✅ Encoded 47 counties and 74 sectors.


# 4. Define Features, Split and Scale
**What this cell does:** Selects the 22 real-world features, splits the data 80/20 (stratified to keep the class balance), and scales the numbers using StandardScaler (required for Logistic Regression).

In [7]:
features = [
    'county_encoded', 'sector_encoded',
    'male_working_owners', 'female_working_owners',
    'total_monthly_expenses', 'monthly_rent_expense', 'monthly_electricity_expense',
    'monthly_credit_expense', 'monthly_social_responsibility_expense',
    'revenue_last_month', 'normal_monthly_revenue', 'net_income_last_month',
    'stock_value_beginning', 'stock_value_end', 'total_turnover_2015',
    'net_income_margin', 'revenue_change_ratio',
    'business_closed', 'number_closed_establishments',
    'revenue_decline', 'zero_or_missing_net_income', 'low_revenue'
]

X = df[features]
y = df['distress_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train set: {X_train_scaled.shape} | Test set: {X_test_scaled.shape}")

✅ Train set: (12651, 22) | Test set: (3163, 22)


# 5. Train the Baseline Model
**What this cell does:** Trains the Logistic Regression model on the scaled real data. `max_iter=5000` gives the algorithm enough rounds to converge on this messy real-world data.

In [8]:
model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train_scaled, y_train)

print("✅ Logistic Regression baseline trained on REAL KNBS data!")

✅ Logistic Regression baseline trained on REAL KNBS data!


# 6. Evaluate the Model
**What this cell does:** Tests the model on the unseen 20% test set and prints the 5 capstone metrics (Accuracy, Precision, Recall, F1, ROC-AUC) plus the full classification report.

In [9]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("="*48)
print("  REAL DATA — LOGISTIC REGRESSION BASELINE")
print("="*48)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print("="*48)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

  REAL DATA — LOGISTIC REGRESSION BASELINE
Accuracy:  0.6532
Precision: 0.5736
Recall:    0.1957
F1-Score:  0.2918
ROC-AUC:   0.6331

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.92      0.77      2008
           1       0.57      0.20      0.29      1155

    accuracy                           0.65      3163
   macro avg       0.62      0.56      0.53      3163
weighted avg       0.63      0.65      0.60      3163



# 7. Save the Model and Preprocessors
**What this cell does:** Saves the trained baseline model, the scaler, and both label encoders. These OVERWRITE the old synthetic versions on purpose — from now on, the app and playground will use the REAL data model.

In [10]:
os.makedirs(os.path.join(base_dir, 'models', 'trained', 'on_real_data'), exist_ok=True)
os.makedirs(os.path.join(base_dir, 'models', 'preprocessing', 'on_real_data'), exist_ok=True)

joblib.dump(model, os.path.join(base_dir, 'models', 'trained', 'on_real_data', 'logistic_regression_baseline.joblib'))
joblib.dump(scaler, os.path.join(base_dir, 'models', 'preprocessing', 'on_real_data', 'scaler.joblib'))
joblib.dump(le_county, os.path.join(base_dir, 'models', 'preprocessing', 'on_real_data', 'le_county.joblib'))
joblib.dump(le_sector, os.path.join(base_dir, 'models', 'preprocessing', 'on_real_data','le_sector.joblib'))

print("✅ Baseline model + preprocessors saved (real-data versions).")

✅ Baseline model + preprocessors saved (real-data versions).
